# Tutorial: Hyperparameter Optimization

> **Type:** Tutorial — follow along step by step. Your only job is to make it work.  
> **Time:** ~1.5 hours  
> **Prerequisites:** Modules 12 (ensemble methods), 13 (cross-validation), 16 (model evaluation)  
> **What you'll build:** A systematic hyperparameter search using three strategies — manual, GridSearchCV, RandomizedSearchCV, and Optuna — on XGBoost, with a final comparison table.

---

## What We're Building

We'll train XGBoost on a synthetic classification dataset and optimise it four ways: by hand, with grid search, random search, and Bayesian optimisation (Optuna). We'll track the compute cost and performance of each approach — so you can see empirically why random search replaced grid search in practice, and why Optuna goes further.

## Step 0: Setup

In [ ]:
# !pip install scikit-learn xgboost optuna matplotlib numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # quiet output
import time

SEED = 42
np.random.seed(SEED)
plt.rcParams['figure.figsize'] = (12, 5)
print('Setup complete.')

## Step 1: Create a Dataset and Baseline

In [ ]:
# Moderately hard classification problem — not linearly separable
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=5, n_clusters_per_class=3,
    random_state=SEED
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Baseline: XGBoost with default hyperparameters
baseline_model = xgb.XGBClassifier(
    random_state=SEED, eval_metric='logloss', verbosity=0
)
baseline_scores = cross_val_score(
    baseline_model, X_train, y_train, cv=5, scoring='roc_auc'
)
baseline_auc = baseline_scores.mean()

print(f"Dataset: {X.shape}")
print(f"Baseline XGBoost ROC-AUC (5-fold CV): {baseline_auc:.4f} ± {baseline_scores.std():.4f}")

## Step 2: Manual Search — Vary One Parameter at a Time

This is what most people do before they know better. It's slow, misses interactions, and is almost never optimal.

In [ ]:
n_estimators_values = [50, 100, 200, 300, 500]
manual_scores = []

for n in n_estimators_values:
    m = xgb.XGBClassifier(n_estimators=n, random_state=SEED, 
                            eval_metric='logloss', verbosity=0)
    score = cross_val_score(m, X_train, y_train, cv=5, scoring='roc_auc').mean()
    manual_scores.append(score)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_estimators_values, manual_scores, marker='o', linewidth=2, color='#4E79A7')
ax.axhline(baseline_auc, color='grey', linestyle='--', label=f'Baseline ({baseline_auc:.4f})')
ax.set_xlabel('n_estimators')
ax.set_ylabel('ROC-AUC (5-fold CV)')
ax.set_title('Manual Search: Vary n_estimators Only', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

best_manual = max(manual_scores)
print(f"Best manual (n_estimators only): {best_manual:.4f}")
print(f"Trials used: {len(n_estimators_values)} (but we only searched 1 of ~10 hyperparameters)")

> **What just happened?** We found a decent n_estimators but completely ignored learning rate, max_depth, subsample, colsample_bytree, and regularisation — all of which interact. Manual one-at-a-time search misses these interactions entirely.

## Step 3: Grid Search

In [ ]:
param_grid = {
    'n_estimators':  [100, 300],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
}
# 2 × 3 × 3 = 18 combinations × 5 folds = 90 model fits
total_fits = 2 * 3 * 3 * 5
print(f"Grid search: {2*3*3} combinations × 5 folds = {total_fits} total fits")

t0 = time.time()
grid_search = GridSearchCV(
    xgb.XGBClassifier(random_state=SEED, eval_metric='logloss', verbosity=0),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1
)
grid_search.fit(X_train, y_train)
grid_time = time.time() - t0

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")
print(f"Time: {grid_time:.1f}s")

## Step 4: Random Search

Same budget as the grid search, but over a wider search space.

In [ ]:
from scipy.stats import uniform, randint

param_distributions = {
    'n_estimators':     randint(50, 500),
    'max_depth':        randint(2, 10),
    'learning_rate':    uniform(0.01, 0.3),
    'subsample':        uniform(0.5, 0.5),        # 0.5–1.0
    'colsample_bytree': uniform(0.5, 0.5),        # 0.5–1.0
    'reg_alpha':        uniform(0, 2),
    'reg_lambda':       uniform(0.5, 3),
}

# Same number of trials as grid search
n_iter = 18
print(f"Random search: {n_iter} random combinations × 5 folds = {n_iter * 5} total fits")
print(f"But sampling from a MUCH wider space — {len(param_distributions)} hyperparameters instead of 3")

t0 = time.time()
random_search = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=SEED, eval_metric='logloss', verbosity=0),
    param_distributions, n_iter=n_iter, cv=5, scoring='roc_auc',
    random_state=SEED, n_jobs=-1
)
random_search.fit(X_train, y_train)
random_time = time.time() - t0

print(f"\nBest params: {random_search.best_params_}")
print(f"Best CV ROC-AUC: {random_search.best_score_:.4f}")
print(f"Time: {random_time:.1f}s")

## Step 5: Bayesian Optimisation with Optuna

In [ ]:
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth':        trial.suggest_int('max_depth', 2, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 2.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'random_state':     SEED,
        'eval_metric':      'logloss',
        'verbosity':        0,
    }
    model = xgb.XGBClassifier(**params)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc').mean()
    return score

t0 = time.time()
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=50)
optuna_time = time.time() - t0

print(f"Best params: {study.best_params}")
print(f"Best CV ROC-AUC: {study.best_value:.4f}")
print(f"Time: {optuna_time:.1f}s (50 trials)")

### Step 5b: Visualise the Optuna Search

In [ ]:
# Plot optimization history manually (doesn't require optuna.visualization)
trial_numbers = [t.number for t in study.trials]
trial_values  = [t.value for t in study.trials]
best_so_far   = pd.Series(trial_values).cummax().tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimisation history
axes[0].scatter(trial_numbers, trial_values, alpha=0.4, s=20, color='#4E79A7', label='Trial score')
axes[0].plot(trial_numbers, best_so_far, color='#E15759', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial number')
axes[0].set_ylabel('ROC-AUC (5-fold CV)')
axes[0].set_title('Optuna: Optimisation History', fontweight='bold')
axes[0].legend()

# Learning rate distribution of good vs bad trials
df_trials = study.trials_dataframe()
median_score = df_trials['value'].median()
good = df_trials[df_trials['value'] >= median_score]
bad  = df_trials[df_trials['value'] <  median_score]

axes[1].hist(bad['params_learning_rate'],  bins=15, alpha=0.6, color='#4E79A7', label='Below median')
axes[1].hist(good['params_learning_rate'], bins=15, alpha=0.6, color='#E15759', label='Above median')
axes[1].set_xlabel('learning_rate')
axes[1].set_ylabel('Count')
axes[1].set_title('Learning Rate: Good vs Bad Trials', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('optuna_search.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6: Final Comparison

In [ ]:
# Train final models with best params and evaluate on held-out test set
def eval_test(model):
    model.fit(X_train, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

baseline_test = eval_test(xgb.XGBClassifier(random_state=SEED, eval_metric='logloss', verbosity=0))
grid_test     = eval_test(xgb.XGBClassifier(**grid_search.best_params_, random_state=SEED, eval_metric='logloss', verbosity=0))
random_test   = eval_test(xgb.XGBClassifier(**random_search.best_params_, random_state=SEED, eval_metric='logloss', verbosity=0))
optuna_test   = eval_test(xgb.XGBClassifier(**study.best_params, random_state=SEED, eval_metric='logloss', verbosity=0))

results = pd.DataFrame([
    {'Method': 'Baseline (defaults)',   'CV AUC': baseline_auc,              'Test AUC': baseline_test, 'Model fits': 5,        'HP dims searched': 0},
    {'Method': 'Manual (1 param)',      'CV AUC': max(manual_scores),        'Test AUC': None,          'Model fits': 5*5,      'HP dims searched': 1},
    {'Method': 'Grid Search',           'CV AUC': grid_search.best_score_,   'Test AUC': grid_test,     'Model fits': total_fits,'HP dims searched': 3},
    {'Method': 'Random Search',         'CV AUC': random_search.best_score_, 'Test AUC': random_test,   'Model fits': n_iter*5, 'HP dims searched': 7},
    {'Method': 'Optuna (Bayesian, 50)', 'CV AUC': study.best_value,          'Test AUC': optuna_test,   'Model fits': 50*5,     'HP dims searched': 7},
])

pd.set_option('display.float_format', '{:.4f}'.format)
print(results.to_string(index=False))

## Step 7: Retrieve Best Parameters and Retrain for Production

In [ ]:
# Use the best configuration found by Optuna
best_params = study.best_params.copy()
best_params.update({'random_state': SEED, 'eval_metric': 'logloss', 'verbosity': 0})

print("Best hyperparameters found:")
for k, v in best_params.items():
    if k not in ('random_state', 'eval_metric', 'verbosity'):
        print(f"  {k:25s}: {v}")

# Train on ALL training data with best params
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

final_test_auc = roc_auc_score(y_test, final_model.predict_proba(X_test)[:, 1])
print(f"\nFinal model test ROC-AUC: {final_test_auc:.4f}")
print(f"Improvement over baseline: +{final_test_auc - baseline_test:.4f}")

---
## What You Built

A complete hyperparameter optimisation pipeline covering four strategies — baseline, manual, grid search, random search, and Bayesian optimisation — with a head-to-head comparison table and a final production-ready model trained on the best configuration found.

## What's Next

- Read `explanation.md` for the theory behind why random beats grid.
- Use `howto.md` as a recipe card when tuning your own models.
- Keep `reference.md` open for hyperparameter ranges of common algorithms.
- **Extension challenge:** Add early pruning to the Optuna study. Use `trial.report(score, step)` and `trial.should_prune()` inside the objective to stop unpromising trials early. Does it improve efficiency?

---

**Next module:** `18_regularization` — because hyperparameter search can improve a model, but regularisation prevents it from overfitting in the first place.